# Módulo 9: Segmentación y Análisis de Contornos
Este módulo extiende los filtros morfológicos del módulo 2 hacia la detección y el análisis de contornos, y hacia dos técnicas de segmentación más avanzadas: Watershed y GrabCut.

In [ ]:
import cv2  # Librería principal de OpenCV: expone las funciones de procesamiento de imágenes y visión por computadora que vamos a usar en todo el módulo
import numpy as np  # NumPy para manejar las imágenes como arrays multidimensionales y operar con las máscaras/matrices que requieren Watershed y GrabCut

In [ ]:
image = cv2.imread("../imgs/img1.jpg")  # Carga la imagen desde disco con cv2.imread; OpenCV la decodifica en formato BGR (no RGB), que es el orden de canales que usa por defecto en toda la librería

In [ ]:
def show_filters(filters):  # Función auxiliar: recibe un diccionario {nombre_de_ventana: imagen} y muestra cada imagen en su propia ventana, una a la vez

    for filter_name, filtered_image in filters.items():  # Itera sobre cada par (nombre, imagen) del diccionario recibido

        cv2.imshow(filter_name, filtered_image)  # Abre una ventana titulada `filter_name` y muestra `filtered_image` en ella

        cv2.waitKey(0)  # Pausa la ejecución hasta que se presione una tecla; el 0 significa "esperar indefinidamente", así se puede inspeccionar cada resultado con calma
        cv2.destroyAllWindows()  # Cierra todas las ventanas abiertas antes de pasar a mostrar la siguiente imagen del diccionario

## Detección de contornos
Convertimos la imagen a escala de grises, la binarizamos con un threshold y usamos `cv2.findContours` para obtener los contornos. Con `cv2.drawContours` los dibujamos todos sobre una copia de la imagen original.

In [ ]:
gray_image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)  # Convierte la imagen a escala de grises: cv2.findContours necesita una imagen de un solo canal (no color)
_, thresh = cv2.threshold(gray_image, 127, 255, cv2.THRESH_BINARY)  # Binariza con un umbral fijo en 127: los píxeles con intensidad >127 pasan a blanco (255) y el resto a negro (0), dejando una imagen de solo dos valores apta para detectar contornos

contours, hierarchy = cv2.findContours(thresh, cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)  # Detecta los contornos (bordes de las regiones blancas) sobre la imagen binaria. RETR_TREE reconstruye la jerarquía completa de contornos anidados (por ejemplo un agujero dentro de una figura); CHAIN_APPROX_SIMPLE comprime los puntos redundantes en tramos rectos horizontales/verticales/diagonales, ahorrando memoria frente a guardar cada píxel del borde

contours_image = image.copy()  # Copia la imagen original a color para dibujar encima sin modificar `image`, que se sigue usando más adelante
cv2.drawContours(contours_image, contours, -1, (0, 255, 0), 2)  # Dibuja los contornos sobre la copia: el -1 indica "todos los contornos de la lista", (0, 255, 0) es verde en BGR y 2 es el grosor de línea en píxeles

print(f"Contornos detectados: {len(contours)}")
show_filters({"Contornos detectados": contours_image})

## Análisis de forma
Para cada contorno relevante (filtramos los que tienen un área muy chica, que suelen ser ruido) usamos `cv2.approxPolyDP` para aproximar un polígono y `cv2.boundingRect` para obtener su caja delimitadora. La cantidad de vértices del polígono aproximado nos sirve como heurística simple para clasificar la forma (triángulo, cuadrado/rectángulo, o círculo aproximado cuando hay muchos vértices). En una foto sin figuras geométricas puras esta clasificación es solo orientativa: lo más confiable es el bounding box y el área de cada contorno.

In [ ]:
min_area = 500  # Umbral de área mínima (en píxeles²) para descartar contornos que suelen ser ruido de la binarización (puntos sueltos, bordes irregulares) en vez de figuras reales

analysis_image = image.copy()  # Copia de la imagen original para dibujar las cajas y etiquetas sin alterar `image`

for contour in contours:
    area = cv2.contourArea(contour)  # Calcula el área encerrada por el contorno (en píxeles²) usando la fórmula de Green sobre los puntos del contorno
    if area < min_area:
        continue  # Descarta contornos chicos: son ruido de la binarización, no aportan a la clasificación de formas

    perimeter = cv2.arcLength(contour, True)  # Calcula el perímetro del contorno; el True indica que es un contorno cerrado (une el último punto con el primero)
    approx = cv2.approxPolyDP(contour, 0.02 * perimeter, True)  # Aproxima el contorno a un polígono más simple con el algoritmo de Douglas-Peucker: reduce la cantidad de puntos manteniendo la forma general, dentro de una tolerancia del 2% del perímetro (a mayor tolerancia, menos vértices en el resultado)
    x, y, w, h = cv2.boundingRect(contour)  # Obtiene el rectángulo delimitador (bounding box) alineado a los ejes que contiene completamente al contorno

    vertices = len(approx)  # Cantidad de vértices del polígono aproximado: se usa como heurística simple para clasificar la forma
    if vertices == 3:
        shape = "Triangulo"
    elif vertices == 4:
        shape = "Cuadrado/Rectangulo"
    elif vertices > 8:
        shape = "Circulo (aprox.)"  # Muchos vértices ~ un polígono que se acerca a una curva suave, se aproxima visualmente a un círculo
    else:
        shape = f"Poligono ({vertices} lados)"

    cv2.rectangle(analysis_image, (x, y), (x + w, y + h), (255, 0, 0), 2)  # Dibuja el bounding box del contorno en azul (BGR), con grosor 2
    cv2.putText(analysis_image, f"{shape} area:{int(area)}", (x, max(y - 10, 0)),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 1)  # Escribe la forma detectada y el área justo encima del box, en rojo; max(y - 10, 0) evita coordenadas negativas cuando el box está pegado al borde superior de la imagen

show_filters({"Analisis de forma": analysis_image})

## Segmentación con Watershed
Watershed necesita marcadores de fondo y de primer plano seguros. Los obtenemos con una binarización Otsu, una apertura morfológica y la transformada de distancia. `cv2.watershed` completa las regiones desconocidas y marca los bordes finales entre regiones con -1.

In [ ]:
_, thresh_otsu = cv2.threshold(gray_image, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)  # Umbral automático con el método de Otsu: en vez de fijar el threshold a mano, lo calcula analizando el histograma de la imagen para maximizar la separación entre dos clases de píxeles; THRESH_BINARY_INV invierte el resultado para que el primer plano quede en blanco (255) y el fondo en negro

kernel = np.ones((3, 3), np.uint8)  # Elemento estructurante 3x3 (matriz de unos) que definen la "forma" de barrido para las operaciones morfológicas siguientes
opening = cv2.morphologyEx(thresh_otsu, cv2.MORPH_OPEN, kernel, iterations=2)  # Apertura morfológica (erosión seguida de dilatación) aplicada 2 veces: elimina puntos de ruido pequeños tipo "sal" sin deformar demasiado el contorno de los objetos grandes

sure_bg = cv2.dilate(opening, kernel, iterations=3)  # Dilata la máscara limpia para obtener una región de "fondo seguro": todo lo que queda FUERA de esta zona dilatada es con certeza fondo, nunca parte de un objeto

dist_transform = cv2.distanceTransform(opening, cv2.DIST_L2, 5)  # Transformada de distancia: para cada píxel de primer plano calcula la distancia euclidiana (DIST_L2, con máscara de 5x5) al píxel de fondo más cercano. Los píxeles más "adentro" de cada blob (lejos de cualquier borde) tienen valores altos, lo que permite ubicar el centro de cada objeto y así separar objetos que se tocan entre sí
_, sure_fg = cv2.threshold(dist_transform, 0.5 * dist_transform.max(), 255, 0)  # Se queda solo con los píxeles cuya distancia supera el 50% del máximo observado: son el "núcleo" de cada objeto, la región de primer plano segura (no incluye los bordes, que son ambiguos)
sure_fg = np.uint8(sure_fg)  # Convierte a uint8 porque connectedComponents y las operaciones siguientes de OpenCV esperan ese tipo de dato

unknown = cv2.subtract(sure_bg, sure_fg)  # Resta las dos máscaras: lo que queda es la región "desconocida", ni fondo seguro ni primer plano seguro; son justamente los bordes entre objetos que Watershed tiene que resolver

_, markers = cv2.connectedComponents(sure_fg)  # Etiqueta cada blob conectado del primer plano seguro con un número entero distinto: estas etiquetas son los "marcadores" (semillas) que le indican a Watershed dónde arranca cada objeto individual
markers = markers + 1  # connectedComponents etiqueta el fondo como 0; se le suma 1 a todo para que el fondo pase a ser 1, dejando libre el valor 0 para marcar la región desconocida en el siguiente paso
markers[unknown == 255] = 0  # Marca explícitamente la región desconocida con 0: es la señal que cv2.watershed interpreta como "zona sin asignar, hay que resolverla"

watershed_image = image.copy()
markers = cv2.watershed(watershed_image, markers)  # Ejecuta el algoritmo Watershed: trata la imagen como un relieve topográfico (según intensidad/gradiente) e "inunda" desde cada marcador conocido hacia afuera; donde dos regiones que crecen se encuentran, traza ahí la línea divisoria y la marca con -1 en `markers`
watershed_image[markers == -1] = [0, 0, 255]  # Pinta de rojo (BGR) todos los píxeles marcados como frontera (-1) entre regiones, para visualizar dónde separó Watershed cada objeto

colored_markers = cv2.applyColorMap(np.uint8(markers % 180), cv2.COLORMAP_JET)  # Aplica un mapa de color falso (JET) para visualizar cada región con un color distinto; el módulo 180 evita desbordar el rango de uint8 (0-255), ya que los IDs de `markers` pueden superar ese valor y provocarían un wrap-around inconsistente sin él

show_filters({
    "Watershed - bordes entre regiones": watershed_image,
    "Watershed - regiones coloreadas": colored_markers
})

## Segmentación con GrabCut
GrabCut estima el primer plano a partir de un rectángulo inicial que aproxima dónde está el objeto de interés. Internamente refina la estimación con modelos de mezcla gaussianos para fondo y primer plano a lo largo de varias iteraciones.

In [ ]:
mask = np.zeros(image.shape[:2], np.uint8)  # Máscara de salida que cv2.grabCut va a rellenar con las etiquetas de segmentación; mismo alto/ancho que la imagen, inicializada en 0

bgd_model = np.zeros((1, 65), np.float64)  # Array temporal que grabCut usa internamente como buffer de trabajo para el modelo de mezcla de gaussianas (GMM) del fondo; el tamaño (1, 65) es un requisito interno del algoritmo, se pasa vacío y la función lo completa
fgd_model = np.zeros((1, 65), np.float64)  # Igual que el anterior pero para el modelo de primer plano; ambos deben pasarse aunque estén vacíos, grabCut los usa y modifica durante las iteraciones

height, width = image.shape[:2]
rect = (int(width * 0.1), int(height * 0.1), int(width * 0.8), int(height * 0.8))  # Rectángulo inicial (x, y, w, h) que delimita aproximadamente dónde está el objeto de interés: acá el 80% central de la imagen, dejando un margen del 10% de cada lado como fondo seguro inicial

cv2.grabCut(image, mask, rect, bgd_model, fgd_model, 5, cv2.GC_INIT_WITH_RECT)  # Ejecuta GrabCut con 5 iteraciones de refinamiento: dentro del rectángulo estima primer plano/fondo probable ajustando los GMMs iterativamente. GC_INIT_WITH_RECT indica que la inicialización parte del rectángulo (todo lo de afuera se asume fondo seguro, lo de adentro primer plano probable a refinar), a diferencia de inicializar con una máscara manual pintada por el usuario

mask2 = np.where((mask == 2) | (mask == 0), 0, 1).astype("uint8")  # grabCut etiqueta cada píxel de `mask` con 0=fondo seguro, 1=primer plano seguro, 2=fondo probable, 3=primer plano probable; acá se arma una máscara binaria que vale 1 solo donde es primer plano (seguro o probable) y 0 en el resto
grabcut_result = image * mask2[:, :, np.newaxis]  # Multiplica la imagen original por la máscara binaria (expandida a 3 canales con newaxis para que el broadcast funcione) dejando en negro todo lo que no es primer plano, extrayendo así el objeto segmentado

show_filters({"GrabCut - primer plano extraido": grabcut_result})

## 🧪 Práctica
Reforzá lo aprendido en este módulo resolviendo los ejercicios guiados en [`practicas/9_practica.ipynb`](../practicas/9_practica.ipynb).